# Classes and encapsulation

Classes become truly useful once you stop seeing them as “functions with extra syntax” and start seeing them as a way to protect and organise state. Encapsulation means that an object controls how its internal data is created, read, and changed.

In Python, encapsulation is more about clear conventions and controlled interfaces than about hard privacy barriers. Properties, methods, and careful attribute design help you express what outside code is allowed to depend on.

A strong question to ask in this module is: **what state should this object own, and what operations should be the only safe way to change it?**

## Visual model

```text
object state + methods that protect that state
```

## How to use this notebook

Read the concept notes first, then run the code cells one at a time. After each run, change an input, prediction, or line of code and rerun it. Intermediate Python becomes easier when you treat every notebook as a place to test a mental model, not just a place to read finished answers.

---

**How to work through this.** Each task below is its own cell. Run them one at a
time and read the output before moving on; that is the whole advantage of a
notebook over a script. Where a cell asks for a prediction, write it before you
run anything. Being wrong on purpose in a place where it costs nothing is how
the correct model gets built.


---

# The concepts behind this exercise

Read this before the tasks. Every idea the tasks below use is explained here, so
you should not need to leave this notebook.

The code cells in this part are demonstrations rather than exercises. Run them,
change a value, run them again. That is the whole point of having them here
instead of in a document.

## Concept 2. The attribute-lookup ladder

**The single most useful diagram in Part 2.** When you write `obj.x`, Python:

```text
1. type(obj).__mro__  -- looking for a DATA DESCRIPTOR named x
                         (something with __get__ AND __set__ -- e.g. @property)
                         found? call its __get__ and STOP.
2. obj.__dict__['x']  -- the instance's own dictionary
                         found? return it and STOP.
3. type(obj).__mro__  -- the class and its bases, in MRO order
                         found? return it (binding it if it is a function)
4. type(obj).__getattr__('x')   -- last-resort hook, if defined
5. AttributeError
```


Two consequences that explain a great deal:

**Instance attributes shadow class attributes** (step 2 beats step 3) — but
**properties beat instance attributes** (step 1 beats step 2). That ordering is
what makes `@property` able to intercept an attribute that used to be plain
data.

**A method is found on the class, not the instance.** Every instance of a class
shares one function object; the binding happens at lookup time.

In [ ]:
class Dog:
    def speak(self): return "woof"

d = Dog()
Dog.speak            # <function Dog.speak>       -- a plain function
d.speak              # <bound method Dog.speak>   -- function + instance
d.speak()            # == Dog.speak(d)

That is all `self` is: the first parameter, filled in by the binding. Python
makes it explicit rather than implicit, which is why you can do this:

In [ ]:
Dog.speak(d)                      # call it unbound
handler = d.speak                 # store a bound method as a callback
list(map(str.upper, ["a", "b"]))  # use an unbound method as a function

---

## Concept 5. `@property`: why Python has no getters

In Java you write getters from the start because changing a public field to a
method later breaks every caller. **In Python it does not**, because
`@property` intercepts attribute access at the same syntax.

In [ ]:
class Circle:
    def __init__(self, radius: float) -> None:
        self.radius = radius      # start plain. No getter, no setter.

    @property
    def area(self) -> float:      # a computed, read-only attribute
        return 3.14159 * self.radius ** 2

c = Circle(2)
c.area                            # 12.56...   -- no parentheses
c.area = 5                        # AttributeError: property has no setter

Adding validation later, without changing any call site:

In [ ]:
class Circle:
    def __init__(self, radius: float) -> None:
        self.radius = radius      # this now goes through the setter

    @property
    def radius(self) -> float:
        return self._radius

    @radius.setter
    def radius(self, value: float) -> None:
        if value <= 0:
            raise ValueError(f"radius must be positive, got {value}")
        self._radius = value

Every existing `c.radius` and `c.radius = 5` keeps working, now validated. This
is why **you should not write a getter and setter until you need one.** Start
with a plain attribute; promote it to a property when there is a reason.

Two things to watch:

**Infinite recursion.** Inside the property, use `self._radius`, never
`self.radius` — the latter calls the property again.

**Cheapness.** A property looks like an attribute, so callers assume it is
cheap. A property that issues a database query will be called in a loop by
someone who had no way to know. If it is expensive, make it a method named
`compute_x()`, or cache it:

In [ ]:
from functools import cached_property

class Dataset:
    @cached_property
    def stats(self) -> dict[str, float]:      # computed once, then stored
        return expensive_analysis(self.rows)  # in the instance __dict__

`cached_property` works by writing the result into `self.__dict__`, so step 2 of
the lookup ladder finds it on every subsequent access and the descriptor never
runs again. (Which means it needs a `__dict__` — it does not work with
`__slots__`.)

---

## Concept 6. `@classmethod` and `@staticmethod`

In [ ]:
class Temperature:
    def __init__(self, kelvin: float) -> None:
        self.kelvin = kelvin

    @classmethod
    def from_celsius(cls, c: float) -> "Temperature":
        return cls(c + 273.15)             # cls, not Temperature

    @classmethod
    def from_fahrenheit(cls, f: float) -> "Temperature":
        return cls.from_celsius((f - 32) * 5 / 9)

    @staticmethod
    def is_valid_kelvin(value: float) -> bool:
        return value >= 0                   # no self, no cls

**`@classmethod` is how Python does named constructors.** A class can have only
one `__init__`, so alternative constructors become classmethods. Using `cls`
rather than the class name means subclasses get the right type back:

In [ ]:
class Kelvin(Temperature): ...
Kelvin.from_celsius(0)          # a Kelvin, not a Temperature

**`@staticmethod` is a function that lives in the class's namespace.** It gets
neither `self` nor `cls`. If it does not use either, ask whether it should be a
module-level function — often the honest answer is yes. It earns its place when
the grouping genuinely aids discovery, or when subclasses should be able to
override it.

---

## Concept 8. Encapsulation that actually works

Since `private` does not exist, encapsulation in Python is about **not handing
out mutable internals** — the Module 02 lesson, applied to design.

In [ ]:
class Playlist:
    def __init__(self, tracks: list[str]) -> None:
        self._tracks = list(tracks)          # copy IN

    @property
    def tracks(self) -> tuple[str, ...]:
        return tuple(self._tracks)           # immutable view OUT

    def add(self, track: str) -> None:
        self._tracks.append(track)

Without the copy on the way in, the caller keeps a handle on your internal list.
Without the conversion on the way out, anyone can mutate it. The underscore
documents intent; the copies enforce it.

The alternatives, each with a trade-off:

| Return | Cost | Caller can |
|---|---|---|
| `tuple(self._tracks)` | O(n) copy | read, index, not mutate |
| `list(self._tracks)` | O(n) copy | mutate their own copy |
| `iter(self._tracks)` | O(1) | iterate once; sees later mutations |
| `MappingProxyType(d)` | O(1) | read a dict; a live view, not a snapshot |
| `self._tracks` | O(1) | **everything.** Not encapsulation. |

---

---

# Now the exercise

You have everything you need. Work top to bottom, and where a cell asks for a
prediction, write it before you run anything.

## The concepts this exercise uses

These are the numbered sections of [the module README](../README.md). If a task below stops making sense, the section named next to it is the one to re-read.

- Section 1: A class body is executable code
- Section 2: The attribute-lookup ladder
- Section 3: Class attributes versus instance attributes
- Section 4: There is no `private`
- Section 5: `@property`: why Python has no getters
- Section 6: `@classmethod` and `@staticmethod`
- Section 7: `__slots__`
- Section 8: Encapsulation that actually works

> The teaching for this module currently lives in the README rather than in this notebook. Read it alongside these cells.

## Setup

Run this first. It is the imports and any shared values the tasks below need.

In [ ]:
from __future__ import annotations

from datetime import date, timedelta
from decimal import Decimal

---

## `Product`

_Product_

In [ ]:
class Product:
    def __init__(self, sku: str, name: str, price: str, cost: str,
                 stock: int, restock_date: date) -> None:
        self._sku = sku
        self._name = name
        self._price = Decimal(price)
        self._cost = Decimal(cost)
        self._stock = stock
        self._restock_date = restock_date
        self._discount_pct = 0

    # -- eight getter/setter pairs, seven of which do nothing --------------
    def get_sku(self) -> str:
        return self._sku

    def get_name(self) -> str:
        return self._name

    def set_name(self, value: str) -> None:
        self._name = value

    def get_price(self) -> Decimal:
        return self._price

    def set_price(self, value: Decimal) -> None:
        if value < 0:
            raise ValueError("price cannot be negative")     # a REAL invariant
        self._price = value

    def get_cost(self) -> Decimal:
        return self._cost

    def set_cost(self, value: Decimal) -> None:
        self._cost = value

    def get_stock(self) -> int:
        return self._stock

    def set_stock(self, value: int) -> None:
        if value < 0:
            raise ValueError("stock cannot be negative")     # a REAL invariant
        self._stock = value

    def get_restock_date(self) -> date:
        return self._restock_date

    def set_restock_date(self, value: date) -> None:
        self._restock_date = value

    def get_discount_pct(self) -> int:
        return self._discount_pct

    def set_discount_pct(self, value: int) -> None:
        if not 0 <= value <= 100:
            raise ValueError("discount must be 0-100")       # a REAL invariant
        self._discount_pct = value

    # -- computed values, currently methods --------------------------------
    def get_margin(self) -> Decimal:
        return self._price - self._cost

    def get_sale_price(self) -> Decimal:
        return self._price * (100 - self._discount_pct) / 100

    def is_in_stock(self) -> bool:
        return self._stock > 0

    def days_until_restock(self) -> int:
        return (self._restock_date - date.today()).days

---

## `ProductV2`

_ProductV2_

In [ ]:
class ProductV2:
    ...

---

## `existing_callers`

Simulates code elsewhere in the codebase that must keep working.

In [ ]:
def existing_callers(p) -> dict[str, object]:  # type: ignore[no-untyped-def]
    """Simulates code elsewhere in the codebase that must keep working.

    Note these use ATTRIBUTE syntax. If your rewrite is correct, the same
    function works against ProductV2 unchanged -- which is the demonstration.
    """
    p.name = "Renamed"
    p.price = Decimal("19.99")
    p.stock = 5
    p.discount_pct = 10
    return {
        "sku": p.sku,
        "name": p.name,
        "price": p.price,
        "margin": p.margin,
        "sale_price": p.sale_price,
        "in_stock": p.in_stock,
    }

---

## `verify`

_verify_

In [ ]:
def verify() -> None:
    p = ProductV2("SKU-1", "Widget", "10.00", "4.00", 3,
                  date.today() + timedelta(days=7))

    result = existing_callers(p)
    assert result["sku"] == "SKU-1"
    assert result["name"] == "Renamed"
    assert result["margin"] == Decimal("15.99")
    assert result["sale_price"] == Decimal("17.991")
    assert result["in_stock"] is True

    for attr, bad in [("price", Decimal("-1")), ("stock", -1),
                      ("discount_pct", 101)]:
        try:
            setattr(p, attr, bad)
        except ValueError:
            pass
        else:
            raise AssertionError(f"{attr} accepted {bad!r}")

    try:
        p.sku = "SKU-2"
    except AttributeError:
        pass
    else:
        raise AssertionError("sku must be read-only")

    assert p.days_until_restock(date.today()) == 7
    assert p.days_until_restock(date.today() + timedelta(days=3)) == 4

    import inspect
    src = inspect.getsource(ProductV2)
    assert "def get_" not in src, "no getters"
    assert src.count("@property") <= 6, "too many properties; some should be plain"

    print("all checks passed")

---

## Run it

This is what running the original file did. Everything above must have been run first.

In [ ]:
if __name__ == "__main__":
    verify()

---

## Before you move on

- [ ] Every cell above ran, in order, on a fresh kernel.
- [ ] You wrote a prediction before running, wherever one was asked for.
- [ ] You can say in one sentence what each task was actually testing.
- [ ] Anything that surprised you is written down in `PROGRESS.md`.

Compare against the worked answers in `../solutions/` only after your own
attempt runs.